# Chapter 7 — End-to-End AgentCore Evaluation

This notebook walks through the **complete evaluation lifecycle** in a single place:

1. **Deploy** a travel assistant agent to AgentCore Runtime
2. **Create** a custom evaluator (LLM-as-judge)
3. **Invoke** the agent with a set of test prompts
4. **Evaluate** the session with built-in and custom metrics
5. **Inspect** and summarise the results

### The Agent

A travel assistant that can answer questions about:
- ✈️  Flights between cities
- 🏨  Hotel recommendations by budget
- 🌤️  Weather forecasts

It should **refuse** to answer questions outside the travel domain — we will verify this with our custom evaluator.

### Tutorial Details

| Information        | Details                                              |
|:-------------------|:-----------------------------------------------------|
| Agent framework    | Strands Agents SDK                                   |
| LLM model          | Anthropic Claude Haiku 3.5 (via Amazon Bedrock)      |
| Deployment         | AgentCore Runtime (CodeBuild, no local Docker needed)|
| Evaluation type    | On-demand (built-in + custom)                        |
| SDK used           | AgentCore Starter Toolkit                            |
| Complexity         | Beginner                                             |

### Prerequisites
- Python 3.10+
- AWS credentials configured (`aws configure` or environment variables)
- IAM permissions: `bedrock-agentcore:*`, `bedrock-agentcore-control:*`, `ecr:*`, `iam:CreateRole`, `codebuild:*`, `s3:*`, `logs:*`

---
## Step 0 — Install dependencies

In [ ]:
!pip install -r requirements.txt -q
!pip install --upgrade bedrock-agentcore-starter-toolkit -q

---
## Step 1 — Setup: imports and AWS session

In [ ]:
import json
import time
import uuid

import boto3
from boto3.session import Session
from IPython.display import Markdown, display

from bedrock_agentcore_starter_toolkit import Evaluation, Runtime

boto_session = Session()
region = boto_session.region_name
account_id = boto3.client("sts").get_caller_identity()["Account"]

print(f"Region  : {region}")
print(f"Account : {account_id}")

---
## Step 2 — Deploy the travel agent to AgentCore Runtime

We use the AgentCore Starter Toolkit `Runtime` class to:
- Configure the agent (entrypoint, IAM role, ECR repo)
- Build the container image via AWS CodeBuild (no local Docker required)
- Deploy to AgentCore Runtime

> ⏱️ First-time deployment takes ~10 minutes (CodeBuild + ECR push). Subsequent updates are faster.

In [ ]:
runtime = Runtime()

runtime.configure(
    entrypoint="agent_app.py",
    agent_name="ch7_travel_agent",
    requirements_file="requirements.txt",
    region=region,
    auto_create_execution_role=True,
    auto_create_ecr=True,
    idle_timeout=120,
)

launch_result = runtime.launch()
print(f"\nDeployment started: {launch_result}")

In [ ]:
def wait_for_ready(runtime, name="Agent", poll_seconds=15):
    """Poll until the agent runtime reaches a terminal status."""
    terminal = {"READY", "CREATE_FAILED", "UPDATE_FAILED", "DELETE_FAILED"}
    while True:
        status = runtime.status().endpoint["status"]
        print(f"  {name} status: {status}")
        if status in terminal:
            return status
        time.sleep(poll_seconds)

status = wait_for_ready(runtime, name="Travel Agent")
assert status == "READY", f"Deployment failed with status: {status}"
print(f"\n✅ Agent is READY")
print(f"   Agent ID  : {launch_result.agent_id}")
print(f"   Agent ARN : {launch_result.agent_arn}")

---
## Step 3 — Create a custom evaluator

We create a **travel quality** evaluator that scores responses on a 5-point scale and penalises the agent for answering out-of-scope questions.

The evaluator config is stored in `travel_quality_metric.json`.

In [ ]:
eval_client = Evaluation(region=region)

# List available built-in evaluators
print("Built-in evaluators available:")
available = eval_client.list_evaluators()
for e in available.get("evaluators", []):
    print(f"  - {e['evaluatorId']}")

In [ ]:
with open("travel_quality_metric.json") as f:
    eval_config = json.load(f)

custom_evaluator = eval_client.create_evaluator(
    name="travel_response_quality",
    level="TRACE",
    description="Evaluates travel assistant response quality and scope adherence",
    config=eval_config,
)

evaluator_id = custom_evaluator["evaluatorId"]
print(f"✅ Custom evaluator created: {evaluator_id}")

---
## Step 4 — Invoke the agent with test prompts

We send a mix of prompts:
- ✅ In-scope travel questions (should score high)
- ❌ Out-of-scope question (should score low with our custom evaluator)

All prompts share the same `session_id` so we can evaluate the full session.

In [ ]:
session_id = str(uuid.uuid4())
print(f"Session ID: {session_id}\n")

test_prompts = [
    "What flights are available from New York to London?",
    "Can you recommend a mid-range hotel in Paris?",
    "What is the weather like in London right now?",
    "Can you help me write a Python script to sort a list?",  # out-of-scope
]

responses = []
for prompt in test_prompts:
    print(f"📤 Prompt: {prompt}")
    response = runtime.invoke(
        payload={"prompt": prompt},
        session_id=session_id,
    )
    print(f"🤖 Response: {response}\n")
    responses.append({"prompt": prompt, "response": response})

print(f"✅ Invoked {len(test_prompts)} prompts in session {session_id}")

---
## Step 5 — Wait for traces to appear in AgentCore Observability

AgentCore Runtime emits OpenTelemetry traces to CloudWatch. We need to wait ~2 minutes for them to be ingested before running evaluations.

In [ ]:
TRACE_INGESTION_WAIT = 180  # seconds (3 minutes — safe default)

print(f"Waiting {TRACE_INGESTION_WAIT}s for traces to be ingested into AgentCore Observability...")
for remaining in range(TRACE_INGESTION_WAIT, 0, -10):
    print(f"  {remaining}s remaining...")
    time.sleep(10)
print("✅ Ready to evaluate")

---
## Step 6 — Run evaluations

We run three evaluation passes on the same session:

| Pass | Evaluator | Level | What it measures |
|------|-----------|-------|------------------|
| A | `Builtin.GoalSuccessRate` | Session | Did the user achieve their goals? |
| B | `Builtin.Correctness` + `Builtin.Helpfulness` | Trace | Per-turn accuracy and usefulness |
| C | `travel_response_quality` (custom) | Trace | Scope adherence + quality |

### Pass A — Session-level: Goal Success Rate

In [ ]:
goal_results = eval_client.run(
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=["Builtin.GoalSuccessRate"],
)

print("=== Goal Success Rate (session level) ===")
for result in goal_results.results:
    display(Markdown(f"""
**Result:** {result.label} ({result.value})

**Explanation:** {result.explanation}

**Token usage:** {result.token_usage}
"""))

### Pass B — Trace-level: Correctness and Helpfulness

In [ ]:
quality_results = eval_client.run(
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=["Builtin.Correctness", "Builtin.Helpfulness"],
)

print("=== Correctness & Helpfulness (trace level) ===")
for result in quality_results.results:
    display(Markdown(f"""
**Metric:** {result.evaluator_name}
**Result:** {result.label} ({result.value})
**Explanation:** {result.explanation}
"""))
    print("---")

### Pass C — Custom evaluator: Travel Quality & Scope Adherence

This is where we expect the out-of-scope Python question to score **Very Poor (0.0)**.

In [ ]:
custom_results = eval_client.run(
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=[evaluator_id],
)

print("=== Travel Quality — Custom Evaluator (trace level) ===")
for result in custom_results.results:
    emoji = "✅" if result.value >= 0.75 else ("⚠️" if result.value >= 0.5 else "❌")
    display(Markdown(f"""
{emoji} **Result:** {result.label} ({result.value})

**Explanation:** {result.explanation}
"""))
    print("---")

---
## Step 7 — Summary

Aggregate all results into a simple score table.

In [ ]:
all_results = (
    list(goal_results.results)
    + list(quality_results.results)
    + list(custom_results.results)
)

print(f"{'Evaluator':<35} {'Label':<15} {'Score':>6}")
print("-" * 60)
for r in all_results:
    name = getattr(r, "evaluator_name", evaluator_id)
    label = getattr(r, "label", "N/A")
    value = getattr(r, "value", 0)
    bar = "█" * int(value * 10)
    print(f"{name:<35} {label:<15} {value:>5.2f}  {bar}")

numeric_values = [getattr(r, "value", 0) for r in all_results if getattr(r, "value", None) is not None]
if numeric_values:
    avg = sum(numeric_values) / len(numeric_values)
    print("-" * 60)
    print(f"{'Average score':<35} {'':15} {avg:>5.2f}")

---
## Step 8 — Save results to file

In [ ]:
import os

os.makedirs("eval_output", exist_ok=True)

save_results = eval_client.run(
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=["Builtin.Correctness", evaluator_id],
    output="eval_output/chapter7_results.json",
)

print("✅ Results saved to eval_output/chapter7_results.json")

---
## (Optional) Cleanup

Uncomment and run the cell below to delete the agent runtime and free up resources.

In [ ]:
# runtime.delete()
# print("Agent runtime deleted")

---
## What you just did

In a single notebook you:

1. **Deployed** a Strands agent to AgentCore Runtime using CodeBuild (no Docker needed)
2. **Created** a custom LLM-as-judge evaluator with a 5-point scale
3. **Invoked** the agent with in-scope and out-of-scope prompts
4. **Evaluated** the session with built-in metrics (GoalSuccessRate, Correctness, Helpfulness) and your custom metric
5. **Verified** that the custom evaluator correctly penalised the out-of-scope response